# Visualization

**Purpose.** Create the case-study map and inspect preserved model diagnostics.

**Inputs.** Lake/catchment geometries and historical evaluation artifacts.

**Outputs.** Curated figures under `docs/figures/`.

> Historical research notebook. Paths assume the repository layout described in `data/README.md`; generated outputs are intentionally not stored in the notebook.


In [ ]:
from pathlib import Path
import os

start_dir = Path.cwd().resolve()
for candidate in (start_dir, *start_dir.parents):
    if (candidate / 'notebooks').is_dir() and (candidate / 'README.md').is_file():
        os.chdir(candidate)
        break
else:
    raise RuntimeError('Run this notebook from inside the cloned repository.')


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.lines import Line2D

# Paths to shapefile and fire data
lake_shapefile = "Datasets/final/lake_catchments_na_cleaned.shp"
fire_csv = "Datasets/GEE/lake_fire_experience_2002_2019.csv"

# Load data
gdf = gpd.read_file(lake_shapefile)
fire_df = pd.read_csv(fire_csv)

# Merge fire experience data
gdf["Lake_ID"] = gdf["Lake_ID"].astype(int)
fire_df["Lake_ID"] = fire_df["Lake_ID"].astype(int)
merged = gdf.merge(fire_df[["Lake_ID", "Years_with_Fire"]], on="Lake_ID", how="left")
merged["Years_with_Fire"] = merged["Years_with_Fire"].fillna(0)
merged["FireStatus"] = merged["Years_with_Fire"].apply(lambda x: "Burned" if x > 0 else "Unburned")

# Reproject to Web Mercator for basemap
merged = merged.to_crs(epsg=3857)

# Color settings
colors = {"Burned": "#d73027", "Unburned": "#4575b4"}

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
for status in ["Unburned", "Burned"]:
    merged[merged["FireStatus"] == status].plot(
        ax=ax,
        color=colors[status],
        edgecolor='none',
        alpha=0.7
    )

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=5)

# Custom legend using Line2D markers
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Unburned Lakes',
           markerfacecolor=colors["Unburned"], markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Burned Lakes',
           markerfacecolor=colors["Burned"], markersize=10)
]
ax.legend(handles=legend_elements, loc='lower left', title="Lake Type", fontsize=12, title_fontsize=13)

# Title and layout
ax.set_title("Case Study Region: Burned and Unburned Lakes in North America", fontsize=16)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("docs/figures/burned_unburned_lakes_map.png", dpi=300)
plt.show()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, r2_score

# 🔁 Load model from checkpoint
def load_model(model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    input_size = checkpoint["input_size"]
    num_lakes = checkpoint["num_lakes"]
    emb_dim = checkpoint["emb_dim"]
    hidden_size = checkpoint["hidden_size"]

    class LSTMWithLakeEmbeddingFusion(torch.nn.Module):
        def __init__(self, input_size, num_lakes, emb_dim=16, hidden_size=64, num_layers=2):
            super().__init__()
            self.embedding = torch.nn.Embedding(num_lakes, emb_dim)
            self.lstm = torch.nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
            self.fc_lstm = torch.nn.Linear(hidden_size, 32)
            self.fc_emb = torch.nn.Linear(emb_dim, 16)
            self.final = torch.nn.Linear(48, 1)

        def forward(self, x, lake_id):
            lstm_out, _ = self.lstm(x)
            last_hidden = lstm_out[:, -1, :]
            lake_emb = self.embedding(lake_id)
            lstm_feat = torch.relu(self.fc_lstm(last_hidden))
            emb_feat = torch.relu(self.fc_emb(lake_emb))
            return self.final(torch.cat([lstm_feat, emb_feat], dim=1))

    model = LSTMWithLakeEmbeddingFusion(input_size, num_lakes, emb_dim, hidden_size)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

# 🔁 Load test data
def load_test_data(folder):
    X_test = np.load(os.path.join(folder, "X_test.npy"))
    y_test = np.load(os.path.join(folder, "y_test.npy"))
    lake_ids_test = np.load(os.path.join(folder, "lake_ids_test.npy"))
    return X_test, y_test, lake_ids_test

# 📊 Grouped lake plot
def plot_grouped_predictions(model, X_test, y_test, lake_ids_test, scaler_y, device, group_size=5):
    model.to(device)
    X_test_torch = torch.tensor(X_test, dtype=torch.float32).to(device)
    lake_ids_torch = torch.tensor(lake_ids_test, dtype=torch.long).to(device)

    with torch.no_grad():
        preds = model(X_test_torch, lake_ids_torch).cpu().numpy()

    # Unscale
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_pred = scaler_y.inverse_transform(preds.reshape(-1, 1)).ravel()

    # 📦 Sort by lake ID
    sort_idx = np.argsort(lake_ids_test)
    y_true = y_true[sort_idx]
    y_pred = y_pred[sort_idx]
    lake_ids_test_sorted = lake_ids_test[sort_idx]

    # 🎨 Plot in groups
    unique_lakes = np.unique(lake_ids_test_sorted)
    for i in range(0, len(unique_lakes), group_size):
        group = unique_lakes[i:i+group_size]
        indices = np.isin(lake_ids_test_sorted, group)

        plt.figure(figsize=(12, 6))
        plt.plot(y_true[indices], label='True CHLA', marker='o', linewidth=1)
        plt.plot(y_pred[indices], label='Predicted CHLA', marker='x', linewidth=1)
        plt.title(f"CHLA Prediction for Lake IDs: {group.tolist()}")
        plt.xlabel("Sorted Sample Index")
        plt.ylabel("CHLA")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()


def plot_sample_predictions(model, X_test, y_test, lake_ids_test, scaler_y, device, num_samples=5):
    model.eval()
    indices = np.random.choice(len(X_test), size=num_samples, replace=False)

    X_sample = torch.tensor(X_test[indices], dtype=torch.float32).to(device)
    lake_id_sample = torch.tensor(lake_ids_test[indices], dtype=torch.long).to(device)
    y_true = y_test[indices]

    with torch.no_grad():
        y_pred_scaled = model(X_sample, lake_id_sample).cpu().numpy().reshape(-1, 1)

    y_pred = scaler_y.inverse_transform(y_pred_scaled).ravel()
    y_true = scaler_y.inverse_transform(y_true.reshape(-1, 1)).ravel()

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(y_true, label="True CHLA", marker="o", linestyle="-")
    ax.plot(y_pred, label="Predicted CHLA", marker="x", linestyle="--")
    ax.set_title(f"Sample Predictions (n={num_samples})")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("CHLA")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_sample_predictions(
    model,
    X_test=X_test,
    y_test=y_test,
    lake_ids_test=lake_ids_test,
    scaler_y=scaler_y_f,  # or scaler_y_nf depending on dataset
    device=device,
    num_samples=200          # you can set to any small value like 5 or 6
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from sklearn.preprocessing import StandardScaler

# Manually re-fit the scaler using the original fire y values (scaled previously)
y_train_f = np.load("Datasets/LSTM2_Fire/y_train.npy")
scaler_y_f = StandardScaler()
scaler_y_f.fit(y_train_f.reshape(-1, 1))  # Refits based on saved y_train

# Same for no-fire:
y_train_nf = np.load("Datasets/LSTM2_NoFire/y_train.npy")
scaler_y_nf = StandardScaler()
scaler_y_nf.fit(y_train_nf.reshape(-1, 1))


# Fire model
model_path = "Datasets/LSTM_Combined/trained_models/chla_lstm_fire_and_no_fire.pth"
model = load_model(model_path, device)

# Test on fire lakes
X_test, y_test, lake_ids_test = load_test_data("Datasets/LSTM2_Fire")
plot_grouped_predictions(model, X_test, y_test, lake_ids_test, scaler_y_f, device, group_size=4)

# Test on no-fire lakes
X_test_nf, y_test_nf, lake_ids_test_nf = load_test_data("Datasets/LSTM2_NoFire")
plot_grouped_predictions(model, X_test_nf, y_test_nf, lake_ids_test_nf, scaler_y_nf, device, group_size=4)
